# Measure Normal and Depth metrics

### Normals
- for measuring normal reconstruction quality, we use Mean Anuglar Error (MAE), Angular Threshold Accuracy (ATA) for 11.25, 22.5, 30

### Depths
- for measurng depth reconstruction qualtiy, we use AbsRel, RMSE, delta < 1.25, delta < 1.25^2


In [19]:
import os
import sys
import torch
import numpy as np
import open3d as o3d
import imageio
from tqdm import tqdm

def rgb_to_normals_uint8(rgb):
    """RGB [0,255] → 단위 노멀 벡터 map (shape H×W×3)"""
    n = rgb.astype(np.float32) / 255.0     # [0,1]
    n = n * 2.0 - 1.0                     # [−1,1]
    length = np.linalg.norm(n, axis=2, keepdims=True)
    length[length < 1e-6] = 1.0           # divide-by-zero 방지
    return n / length                     # 단위 벡터

# convert all diffren normal convention to stable convention
def convert_diffren_to_stable(normals_d):
    ss = np.empty_like(normals_d)
    ss[..., 0] = normals_d[..., 0]  
    ss[..., 1] = -normals_d[..., 1]  
    ss[..., 2] = -normals_d[..., 2] 
    # (필요하면 다시 정규화)
    norm = np.linalg.norm(ss, axis=2, keepdims=True)
    norm[norm < 1e-6] = 1.0
    return ss / norm


In [57]:
import os
import sys
import numpy as np
from tqdm import tqdm
from PIL import Image
import yaml
from scipy.spatial.transform import Rotation

# --- [수정 불필요] OpenCV 커스텀 YAML 태그 처리 설정 ---
def opencv_matrix_constructor(loader, node):
    mapping = loader.construct_mapping(node, deep=True)
    return mapping
yaml.add_constructor('tag:yaml.org,2002:opencv-matrix', opencv_matrix_constructor, Loader=yaml.SafeLoader)
# ----------------------------------------------------

def qvec2rotmat(qvec):
    """쿼터니언 벡터(w, x, y, z)를 3x3 회전 행렬로 변환합니다."""
    return Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]]).as_matrix()

def process_normal_map(image_path, down_scale=1):
    img = Image.open(image_path).convert("RGB")

    # --- [추가된 부분] 이미지 다운스케일링 ---
    if down_scale > 1:
        original_width, original_height = img.size
        new_width = int(original_width / down_scale)
        new_height = int(original_height / down_scale)
        
        # LANCZOS는 고품질 리샘플링 필터로, 이미지 축소 시 디테일 손실을 최소화합니다.
        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    # -----------------------------------------

    normal = np.array(img).astype(np.float32)
    normal = (normal / 255.0) * 2.0 - 1.0
    
    norm = np.linalg.norm(normal, axis=2, keepdims=True)
    # 0으로 나누는 것을 방지하기 위해 norm이 0인 벡터의 길이를 1로 설정
    norm[norm == 0] = 1.0
    normal /= norm
    
    return normal

def align_camera_poses(pred_centers, gt_centers):
    """Umeyama 알고리즘을 사용해 predicted 카메라 중심을 GT에 정렬하는 회전 행렬을 찾습니다."""
    if pred_centers.shape[0] == 0:
        raise ValueError("Cannot align empty camera sets.")
    mu_pred = pred_centers.mean(axis=0)
    mu_gt = gt_centers.mean(axis=0)
    X = pred_centers - mu_pred
    Y = gt_centers - mu_gt
    H = X.T @ Y
    U, _, Vt = np.linalg.svd(H)
    R_align = Vt.T @ U.T
    if np.linalg.det(R_align) < 0:
        Vt[-1, :] *= -1
        R_align = Vt.T @ U.T
    return R_align

# --- [수정된 함수 1] ---
def read_gt_cameras(path):
    """
    GT 카메라 파일을 파싱합니다.
    - 변경점: 파일 이름 대신 '프레임 번호'를 키로 사용합니다. (예: 'val_0000.png' -> '0000')
    """
    poses = {}
    centers = {}
    keys = []
    with open(path, "r") as f:
        for line in f:
            if line.strip().startswith("#") or len(line.strip()) == 0: continue
            parts = line.strip().split()
            if len(parts) < 10: continue
            try:
                img_name = parts[9]
                # 'val_0000.png' -> '0000' 추출
                key = img_name.split('.')[0].split('_')[-1]
                
                qvec = np.array(tuple(map(float, parts[1:5])))
                tvec = np.array(tuple(map(float, parts[5:8])))
                
                R_c2w = qvec2rotmat(qvec)
                T_c2w = tvec
                center = -R_c2w.T @ T_c2w
                
                poses[key] = (R_c2w, T_c2w)
                centers[key] = center
                keys.append(key)
            except (ValueError, IndexError) as e:
                print(f"경고: 다음 라인을 파싱할 수 없습니다: '{line.strip()}'. 오류: {e}")
                continue
    return poses, centers, sorted(keys)

# --- [수정된 함수 2] ---
def read_pred_cameras(path):
    """
    Predicted 카메라 파일(extri.yml)을 파싱합니다.
    - 변경점: 파일 이름 대신 '프레임 번호'를 키로 사용합니다. (예: '0000')
    """
    poses = {}
    centers = {}
    keys = []
    try:
        with open(path, 'r') as f:
            content = f.read()
        if content.startswith('%YAML:1.0'):
            content = content.split('---', 1)[1]
        data = yaml.safe_load(content)
    except Exception as e:
        print(f"오류: YAML 파일을 읽거나 파싱할 수 없습니다: {path}. 오류: {e}")
        return {}, {}, []

    if 'names' not in data:
        print("오류: YAML 파일에 'names' 키가 존재하지 않습니다.")
        return {}, {}, []

    for key in data['names']: # key는 '0000', '0001' 등
        try:
            rot_data = data[f'Rot_{key}']
            t_data = data[f'T_{key}']
            
            R_c2w = np.array(rot_data['data']).reshape(rot_data['rows'], rot_data['cols'])
            T_c2w = np.array(t_data['data']).flatten()
            center = -R_c2w.T @ T_c2w
            
            poses[key] = (R_c2w, T_c2w)
            centers[key] = center
            keys.append(key)
        except KeyError as e:
            print(f"경고: {key}에 대한 카메라 정보를 찾을 수 없습니다. 키 오류: {e}")
            continue
    return poses, centers, sorted(keys)


In [60]:
 # Define input paths
pred_camera_path = "data/datasets/manual_synthetic/scene_7/extri.yml"
gt_camera_path = "/home/youngju/ssd/datasets/synthetic/scene_7/images.txt"
pred_path = "data/result/envgs/manual_synthetic/0831-manual-synthetic/scene_7"
gt_path   = "/home/youngju/ssd/datasets/synthetic/scene_7"

pred_normal_path = os.path.join(pred_path, "NORMAL")
gt_normal_path   = os.path.join(gt_path, "normals_gt")


print("Reading ground truth camera poses...")
gt_poses, gt_centers, gt_keys = read_gt_cameras(gt_camera_path)

print("Reading predicted camera poses...")
pred_poses, pred_centers, pred_keys = read_pred_cameras(pred_camera_path)

# --- [수정된 로직 1] 공통 키(프레임 번호)를 찾아 데이터 정렬 ---
common_keys = sorted(list(set(gt_keys) & set(pred_keys)))

if not common_keys:
    print("오류: GT와 Predicted 데이터 간에 일치하는 프레임 번호가 없습니다.")
    print(f"GT keys (첫 5개): {gt_keys[:5]}")
    print(f"Pred keys (첫 5개): {pred_keys[:5]}")
    sys.exit(1)

aligned_gt_centers = np.array([gt_centers[key] for key in common_keys])
aligned_pred_centers = np.array([pred_centers[key] for key in common_keys])

print(f"공통 카메라 수: {len(common_keys)}")

print("Aligning camera coordinate systems...")
R_align = align_camera_poses(aligned_pred_centers, aligned_gt_centers)
print("Alignment complete.")

all_angular_errors = []

for key in tqdm(common_keys, desc="Evaluating Normals"):
    # 각 키(프레임 번호)를 사용하여 파일 이름 재구성
    gt_img_name = f"val_normal_{key}.png"
    # 참고: pred 파일 이름이 항상 'frameXXXX_camera0000.png' 형식이라고 가정합니다.
    # 만약 'cameraYYYY' 부분이 바뀐다면 이 부분을 수정해야 합니다.
    pred_img_name = f"frame0000_camera{key}.png"

    pred_normal_file = os.path.join(pred_normal_path, pred_img_name)
    gt_normal_file = os.path.join(gt_normal_path, gt_img_name)
    
    if not os.path.exists(pred_normal_file) or not os.path.exists(gt_normal_file):
        continue

    print(f"Processing frame {key}")

    pred_normal_cam = process_normal_map(pred_normal_file)
    gt_normal_blender = process_normal_map(gt_normal_file, down_scale=2)

    R_pred_c2w, _ = pred_poses[key]
    R_gt_c2w, _ = gt_poses[key]

    R_full_transform = R_gt_c2w.T @ R_align @ R_pred_c2w
    pred_normal_transformed = np.einsum('ij,hwj->hwi', R_full_transform, pred_normal_cam)

    gt_normal_cv = gt_normal_blender.copy()
    gt_normal_cv[..., 1:] *= -1
    
    mask = np.linalg.norm(gt_normal_blender, axis=-1) > 0.5
    
    dot_product = np.sum(pred_normal_transformed[mask] * gt_normal_cv[mask], axis=-1)
    dot_product = np.clip(dot_product, -1.0, 1.0)
    angular_error = np.rad2deg(np.arccos(dot_product))
    
    all_angular_errors.extend(angular_error)

if all_angular_errors:
    mae = np.mean(all_angular_errors)
    print("\n-------------------------------------------")
    print(f"Mean Angular Error (MAE): {mae:.4f} degrees")
    print("-------------------------------------------")
else:
    print("MAE를 계산할 수 없습니다. 처리된 유효한 normal이 없습니다.")

Reading ground truth camera poses...
Reading predicted camera poses...
공통 카메라 수: 160
Aligning camera coordinate systems...
Alignment complete.


Evaluating Normals:   6%|▌         | 9/160 [00:00<00:02, 70.28it/s]

Processing frame 0000
Processing frame 0008
Processing frame 0016
Processing frame 0024


Evaluating Normals:  26%|██▌       | 41/160 [00:00<00:01, 114.65it/s]

Processing frame 0032
Processing frame 0040
Processing frame 0048
Processing frame 0056


...

Processing frame 0088
Processing frame 0096
Processing frame 0104
Processing frame 0112


Evaluating Normals:  76%|███████▌  | 121/160 [00:01<00:00, 120.73it/s]

Processing frame 0120
Processing frame 0128
Processing frame 0136


Evaluating Normals: 100%|██████████| 160/160 [00:01<00:00, 118.13it/s]


Processing frame 0144
Processing frame 0152

-------------------------------------------
Mean Angular Error (MAE): 101.9394 degrees
-------------------------------------------
